In [61]:
# ============================================================
# df_master_dummy builder
# Input: df_saldos (cliente, confidencial) con grano (folio, MOB)
# Output: df_master_dummy con:
#   - cosecha = YYYY-MM (string)
#   - Monto Fondeado (denominador)
#   - BGI2+/3+/4+/5+/CAST_VAL (numeradores por monto)
#   - *_CONTEO_EVER y CAST_CONTEO (EVER)
#   - dimensiones dummy por folio (Perfil, Ciudad, etc.)
#   - Cluster Mora dummy derivado de moras reales (defendible)
# ============================================================

from __future__ import annotations
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

In [ ]:
# ---------------------------
# Catálogos de dimensiones dummy
# (los que ya me compartiste)
# ---------------------------
df_info = {
    'Perfil': ['A', 'B', 'C', 'D', 'E', 'F'],
    'Ciudad Atria': ['Cancún', 'CDMX', 'Celaja',
                     'Chihuahua', 'Ciudad Juarez', 'Guadlajara',
                     'Irapuato', 'León', 'Merida',
                     'Monterrey', 'Pachuca', 'Queretaro',
                     'Torreón'],
    'Plazo': ['12', '24', '36', '48', '60'],
    'Antiguedad Unidad': ['<4', '4 a <6', '6 a <8', '>=8'],
    'Laboratorios': ['W01', 'W2', 'W3', 'Z01', 'Z02', 'Z03', 'Z04', 'Normal'],
    'Notificaciones': ['PE A', 'PE B', 'GG 1', 'GG 2', 'GG 3', 'Normal'],
    'Atria Plus': ['PE', 'GG', 'Score', 'Normal'],
    'MMX 6': ['MOP1',  'MOP2',  'MOP3',  'MOP4',  'MOP5+'],
    'MMX 12': ['MOP1',  'MOP2',  'MOP3',  'MOP4',  'MOP5+'],
    'MMX 24': ['MOP1',  'MOP2',  'MOP3',  'MOP4',  'MOP5+'],
    'Score generico de bancos': 'Entero del 200 al 900',
    'Cluster Mora': ['BGI2+', 'BGI3+', 'BGI4+', 'BGI5+', 'CAST'], 
    'Cluster Experiencia': ['Exp1', 'Exp2', 'Exp3', 'Exp4', 'Exp5'],
    'Score Buro': ['>0 a <556', '556 a <613', '613 a <663', '>663'],
    '% Utilizacion Revolvente': ['NULL', '>=0 a <0.068',
                                 '>=0.068 a <0.305', '>=0.305 a <0.621',
                                 '>=0.621663'],
    'Endeudamiento': ['NULL', '<0.019', '>=0.019 a <0.067',
                      '>=0.067 a <0.171', '>=0.171 a <0.305',
                      '>=0.305'],
}

In [63]:
df = pd.read_excel('Saldo_cosechas_20251201.xlsx')

In [64]:
df.columns

Index(['folio', 'MOB', 'pv', 'Fecha de Inicio', 'Mes-Año', 'Monto Fondeado',
       'Saldo Capital', 'Tipo de Castigo', 'Estatus Legal', 'BGI4+', 'Finan',
       'Mob Castigo', 'Trimestre', 'BGI4+_MAX', 'BGI4+_CONTEO',
       'BGI4+_CONTEO_EVER', 'CONTEO_FINAN', 'BGI2+', 'BGI3+', 'BGI5+',
       'BGI2+_CONTEO', 'BGI3+_CONTEO', 'BGI5+_CONTEO', 'BGI2+_CONTEO_EVER',
       'BGI3+_CONTEO_EVER', 'BGI5+_CONTEO_EVER', 'CAST_VAL', 'CAST_CONTEO',
       'CAST_VAL_ORI'],
      dtype='object')

In [65]:
print(df['Tipo de Castigo'].unique())
print(df['Estatus Legal'].unique())

[nan 'Mora' 'Administrativo' 'Operativo']
[nan 'Mora' 'Cancelación' 'Otros' 'Pérdida total con expediente'
 'Dación en pago' 'Fraude' 'Siniestro' 'Defunción del titular']


In [66]:
df['Fecha de Inicio']

0       2022-06-01
1       2022-06-01
2       2022-06-01
3       2022-06-01
4       2022-06-01
           ...    
69390   2025-10-01
69391   2025-10-01
69392   2025-10-01
69393   2025-10-01
69394   2025-10-01
Name: Fecha de Inicio, Length: 69395, dtype: datetime64[ns]

In [67]:
df_capital = df[(df['Fecha de Inicio'] == '2022-09-01') & (df['MOB'] == 1)].head(10).copy()
df_capital

,folio,MOB,pv,Fecha de Inicio,Mes-Año,Monto Fondeado,Saldo Capital,Tipo de Castigo,Estatus Legal,BGI4+,...,BGI5+,BGI2+_CONTEO,BGI3+_CONTEO,BGI5+_CONTEO,BGI2+_CONTEO_EVER,BGI3+_CONTEO_EVER,BGI5+_CONTEO_EVER,CAST_VAL,CAST_CONTEO,CAST_VAL_ORI
633,111120220800054,1,0,2022-09-01,2022-10-01,95000.00,93311.60,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
905,111120220800080,1,0,2022-09-01,2022-10-01,209545.16,177397.76,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
1177,111120220800116,1,0,2022-09-01,2022-10-01,249760.14,247861.65,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
1215,111120220800118,1,0,2022-09-01,2022-10-01,130000.00,128833.00,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
1253,111120220800122,1,0,2022-09-01,2022-10-01,344717.57,0.00,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
1291,111120220800123,1,0,2022-09-01,2022-10-01,119000.00,117830.64,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
1329,111120220800132,1,0,2022-09-01,2022-10-01,200500.00,198236.39,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
1367,111120220900002,1,0,2022-09-01,2022-10-01,180000.00,178631.77,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
1405,111120220900003,1,0,2022-09-01,2022-10-01,162256.48,161023.12,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
1443,111120220900007,1,0,2022-09-01,2022-10-01,220000.00,218327.72,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN


In [68]:
df_capital[['Monto Fondeado']].sum()

Monto Fondeado    1910779.35
dtype: float64

In [69]:
107677+915900+1881466.32+1910779.35

4815822.67

In [70]:
df[df['BGI2+'] > 0][['Fecha de Inicio', 'BGI2+', 'Saldo Capital']].groupby('Fecha de Inicio').sum('Saldo Capital')

,BGI2+,Saldo Capital
Fecha de Inicio,,
2022-07-01,8.570745e+05,8.570745e+05
2022-08-01,4.576934e+06,4.576934e+06
2022-09-01,6.752011e+06,6.752011e+06
2022-10-01,2.567378e+06,2.567378e+06
2022-11-01,7.926187e+06,7.926187e+06
2022-12-01,1.874582e+07,1.874582e+07
2023-01-01,2.782307e+07,2.782307e+07
2023-02-01,2.492974e+07,2.492974e+07
2023-03-01,5.038833e+07,5.038833e+07


In [71]:
# ---------------------------
# Configurable: nombres de columnas en df_saldos
# Ajusta aquí SI tu df_saldos usa otros nombres
# ---------------------------
@dataclass(frozen=True)
class Cols:
    folio: str = "folio"
    mob: str = "MOB"
    fecha_inicio: str = "Fecha de Inicio"
    monto_fondeado: str = "Monto Fondeado"
    tipo_de_castigo: str = "Tipo de Castigo"
    estatus_legal: str = "Estatus Legal"
    # moras monto
    bgi2: str = "BGI2+"
    bgi3: str = "BGI3+"
    bgi4: str = "BGI4+"
    bgi5: str = "BGI5+"
    cast_val: str = "CAST_VAL"
    # ever
    bgi2_ever: str = "BGI2+_CONTEO_EVER"
    bgi3_ever: str = "BGI3+_CONTEO_EVER"
    bgi4_ever: str = "BGI4+_CONTEO_EVER"
    bgi5_ever: str = "BGI5+_CONTEO_EVER"
    cast_ever: str = "CAST_CONTEO"


REQUIRED_NUM_COLS = [
    "Monto Fondeado", "BGI2+", "BGI3+", "BGI4+", "BGI5+", "CAST_VAL"
]
REQUIRED_EVER_COLS = [
    "BGI2+_CONTEO_EVER", "BGI3+_CONTEO_EVER", "BGI4+_CONTEO_EVER", "BGI5+_CONTEO_EVER", "CAST_CONTEO"
]

In [72]:
# ---------------------------
# Pesos "plausibles" (puedes ajustar)
# ---------------------------
DEFAULT_WEIGHTS: Dict[str, Dict[str, float]] = {
    # perfiles: más masa en B/C/D
    "Perfil": {"A": 0.10, "B": 0.18, "C": 0.22, "D": 0.22, "E": 0.18, "F": 0.10},
    # ciudades: CDMX/MTY más frecuentes (ajusta según tu intuición)
    "Ciudad Atria": {
        "CDMX": 0.20, "Monterrey": 0.14, "Guadlajara": 0.12, "Cancún": 0.06,
        "Queretaro": 0.07, "León": 0.06, "Merida": 0.05, "Chihuahua": 0.05,
        "Ciudad Juarez": 0.05, "Pachuca": 0.05, "Torreón": 0.05,
        "Celaja": 0.05, "Irapuato": 0.05
    },
    # plazos: típicamente 36/48 dominan
    "Plazo": {"12": 0.10, "24": 0.18, "36": 0.30, "48": 0.27, "60": 0.15},
    "Antiguedad Unidad": {"<4": 0.30, "4 a <6": 0.30, "6 a <8": 0.22, ">=8": 0.18},
    "Laboratorios": {"Normal": 0.35, "W01": 0.10, "W2": 0.10, "W3": 0.10, "Z01": 0.08, "Z02": 0.09, "Z03": 0.09, "Z04": 0.09},
    "Notificaciones": {"Normal": 0.35, "PE A": 0.12, "PE B": 0.12, "GG 1": 0.14, "GG 2": 0.14, "GG 3": 0.13},
    "Atria Plus": {"Normal": 0.40, "PE": 0.20, "GG": 0.20, "Score": 0.20},
    "MMX 6": {"MOP1": 0.30, "MOP2": 0.25, "MOP3": 0.20, "MOP4": 0.15, "MOP5+": 0.10},
    "MMX 12": {"MOP1": 0.28, "MOP2": 0.25, "MOP3": 0.21, "MOP4": 0.16, "MOP5+": 0.10},
    "MMX 24": {"MOP1": 0.26, "MOP2": 0.25, "MOP3": 0.22, "MOP4": 0.17, "MOP5+": 0.10},
    "Cluster Experiencia": {"Exp1": 0.20, "Exp2": 0.22, "Exp3": 0.22, "Exp4": 0.20, "Exp5": 0.16},
    "Score Buro": {">0 a <556": 0.25, "556 a <613": 0.25, "613 a <663": 0.25, ">663": 0.25},
    "% Utilizacion Revolvente": {"NULL": 0.15, ">=0 a <0.068": 0.20, ">=0.068 a <0.305": 0.25, ">=0.305 a <0.621": 0.25, ">=0.621663": 0.15},
    "Endeudamiento": {"NULL": 0.15, "<0.019": 0.15, ">=0.019 a <0.067": 0.20, ">=0.067 a <0.171": 0.20, ">=0.171 a <0.305": 0.15, ">=0.305": 0.15},
}

In [73]:
def _normalize_weights(options: List[str], weights: Optional[Dict[str, float]]) -> np.ndarray:
    """Return probabilities aligned to options."""
    if not weights:
        return np.ones(len(options), dtype=float) / len(options)

    w = np.array([weights.get(o, 0.0) for o in options], dtype=float)
    if w.sum() <= 0:
        return np.ones(len(options), dtype=float) / len(options)
    return w / w.sum()


def _weighted_choice(rng: np.random.Generator, options: List[str], weights: Optional[Dict[str, float]], size: int) -> np.ndarray:
    p = _normalize_weights(options, weights)
    return rng.choice(options, size=size, replace=True, p=p)


def _ensure_datetime_month_start(s: pd.Series) -> pd.Series:
    dt = pd.to_datetime(s, errors="coerce")
    if dt.isna().any():
        bad = s[dt.isna()].head(5).tolist()
        raise ValueError(f"No pude convertir a datetime algunos valores en '{s.name}'. Ejemplos: {bad}")
    # month start
    return dt.dt.to_period("M").dt.to_timestamp()


def derive_cluster_mora_dummy_C0C5(
    df: pd.DataFrame,
    cols,
    recent_k: int = 3,
    ever_high: float = 0.60,   # umbral: % de MOBs con EVER=1 para severa
    ever_mid: float = 0.25,    # umbral: % de MOBs con EVER=1 para moderada
) -> pd.Series:
    """
    Cluster Mora dummy (interpretables, C0..C5) folio-level, defendible:

    C0 - Sin mora
    C1 - Mora leve
    C2 - Mora moderada
    C3 - Mora severa
    C4 - CAST
    C5 - Recuperado / Curado (tuvo mora antes pero en los últimos recent_k MOBs no)

    Usa señales reales: CAST_VAL, BGI2+/3+/4+/5+, y EVER.
    """

    # Asegurar orden por MOB para el check de "curado"
    df2 = df[[cols.folio, cols.mob, cols.cast_val, cols.bgi2, cols.bgi3, cols.bgi4, cols.bgi5,
              cols.bgi2_ever, cols.bgi3_ever, cols.bgi4_ever, cols.bgi5_ever, cols.cast_ever]].copy()

    df2 = df2.sort_values([cols.folio, cols.mob])

    g = df2.groupby(cols.folio, sort=False)

    # Señales "alguna vez"
    has_cast = (g[cols.cast_val].max() > 0) | (g[cols.cast_ever].max() > 0)
    has_bgi5 = (g[cols.bgi5].max() > 0) | (g[cols.bgi5_ever].max() > 0)
    has_bgi4 = (g[cols.bgi4].max() > 0) | (g[cols.bgi4_ever].max() > 0)
    has_bgi3 = (g[cols.bgi3].max() > 0) | (g[cols.bgi3_ever].max() > 0)
    has_bgi2 = (g[cols.bgi2].max() > 0) | (g[cols.bgi2_ever].max() > 0)

    # Ever agregado (cualquier mora ever)
    ever_any = (
        df2[cols.bgi2_ever].astype(int)
        | df2[cols.bgi3_ever].astype(int)
        | df2[cols.bgi4_ever].astype(int)
        | df2[cols.bgi5_ever].astype(int)
        | df2[cols.cast_ever].astype(int)
    )
    df2["_ever_any"] = ever_any

    ever_rate = g["_ever_any"].mean()  # proporción de MOBs con ever=1

    # Check "curado": tuvo mora antes, pero en últimos recent_k MOBs no hay mora (ni cast)
    def _is_cured(sub: pd.DataFrame) -> bool:
        # tuvo alguna mora/cast en algún momento:
        ever_hist = sub["_ever_any"].to_numpy()
        if ever_hist.max() == 0:
            return False  # no tuvo mora, no aplica "curado"
        # últimos k:
        tail = sub.tail(recent_k)
        tail_ever = tail["_ever_any"].to_numpy().max() == 0
        tail_cast = (tail[cols.cast_val].fillna(0).to_numpy().max() > 0) or (tail[cols.cast_ever].fillna(0).to_numpy().max() > 0)
        return tail_ever and (not tail_cast)

    cured = g.apply(_is_cured)

    idx = has_cast.index
    out = pd.Series("C0 - Sin mora", index=idx, dtype="object")

    # CAST domina
    out.loc[has_cast] = "C4 - CAST"

    # Curado (solo si NO cast)
    out.loc[(~has_cast) & cured] = "C5 - Recuperado / Curado"

    # Severa/moderada/leve (solo si no cast y no curado)
    mask_free = (~has_cast) & (~cured)

    # Severa por presencia de BGI4/5 o ever alto
    out.loc[mask_free & (has_bgi5 | has_bgi4 | (ever_rate >= ever_high))] = "C3 - Mora severa"

    # Moderada por BGI3 o ever medio
    out.loc[mask_free & ~(has_bgi5 | has_bgi4 | (ever_rate >= ever_high)) & (has_bgi3 | (ever_rate >= ever_mid))] = "C2 - Mora moderada"

    # Leve por solo BGI2
    out.loc[mask_free & ~(has_bgi5 | has_bgi4 | has_bgi3 | (ever_rate >= ever_mid)) & has_bgi2] = "C1 - Mora leve"

    # Sin mora (queda por default)
    return out

In [74]:
def build_df_dim_folio_dummy(
    df_saldos: pd.DataFrame,
    seed: int = 42,
    cols: Cols = Cols(),
    df_info_catalog: Dict[str, object] = df_info,
    weights: Dict[str, Dict[str, float]] = DEFAULT_WEIGHTS,
) -> pd.DataFrame:
    """
    Opción A: df_dim_folio_dummy ES dueño de 'cosecha'.
    Salida: 1 fila por folio con:
      - folio
      - cosecha (YYYY-MM)
      - dimensiones dummy
      - Cluster Mora (C0..C5) derivado de señales reales
    """
    rng = np.random.default_rng(seed)

    # base folio-level con fecha inicio para derivar cosecha (la fecha NO sale)
    base = df_saldos[[cols.folio, cols.fecha_inicio]].drop_duplicates(subset=[cols.folio]).copy()
    cosecha_date = _ensure_datetime_month_start(base[cols.fecha_inicio])
    base["cosecha"] = cosecha_date.dt.strftime("%Y-%m")
    base = base.drop(columns=[cols.fecha_inicio])

    n = len(base)

    def assign_cat(colname: str) -> np.ndarray:
        options = df_info_catalog[colname]
        if not isinstance(options, list):
            raise ValueError(f"El catálogo df_info['{colname}'] debe ser una lista.")
        return _weighted_choice(rng, options, weights.get(colname), size=n)

    cat_cols = [
        "Perfil", "Ciudad Atria", "Plazo", "Antiguedad Unidad",
        "Laboratorios", "Notificaciones", "Atria Plus",
        "MMX 6", "MMX 12", "MMX 24",
        "Cluster Experiencia", "Score Buro",
        "% Utilizacion Revolvente", "Endeudamiento",
    ]
    for c in cat_cols:
        base[c] = assign_cat(c)

    base["Score generico de bancos"] = rng.integers(200, 901, size=n, dtype=np.int16)

    # Cluster Mora (C0..C5), derivado de moras reales (defendible)
    cluster = derive_cluster_mora_dummy_C0C5(df_saldos, cols=cols)
    base["Cluster Mora"] = cluster.reindex(base[cols.folio]).to_numpy()

    keep = [cols.folio, "cosecha"] + cat_cols + ["Score generico de bancos", "Cluster Mora"]
    return base[keep]

In [75]:
def build_df_master_dummy(
    df_saldos: pd.DataFrame,
    seed: int = 42,
    cols: Cols = Cols(),
) -> pd.DataFrame:
    """
    Construye df_master_dummy LIMPIO (sin columnas extra), con Opción A:

    - Fact recortado SOLO a:
        folio, MOB, Monto Fondeado,
        BGI2+/BGI3+/BGI4+/BGI5+/CAST_VAL,
        BGI*_CONTEO_EVER + CAST_CONTEO
    - No se crea 'cosecha' en fact.
    - Se hace merge con df_dim_folio_dummy (que sí trae 'cosecha').
    - Se elimina cualquier columna no contemplada.
    """
    # Validar columnas requeridas
    required = [
        cols.folio, cols.mob, cols.fecha_inicio, cols.monto_fondeado,
        cols.tipo_de_castigo, cols.estatus_legal,
        cols.bgi2, cols.bgi3, cols.bgi4, cols.bgi5, cols.cast_val,
        cols.bgi2_ever, cols.bgi3_ever, cols.bgi4_ever, cols.bgi5_ever, cols.cast_ever,
    ]
    missing = [c for c in required if c not in df_saldos.columns]
    if missing:
        raise ValueError(f"df_saldos no trae columnas requeridas: {missing}")

    # --- 1) Dim folio-level (dueño de cosecha) ---
    df_dim = build_df_dim_folio_dummy(df_saldos, seed=seed, cols=cols)

    # --- 2) Fact recortado (NO arrastra basura) ---
    fact_cols = [
        cols.folio, cols.mob, cols.monto_fondeado,
        cols.tipo_de_castigo, cols.estatus_legal,
        cols.bgi2, cols.bgi3, cols.bgi4, cols.bgi5, cols.cast_val,
        cols.bgi2_ever, cols.bgi3_ever, cols.bgi4_ever, cols.bgi5_ever, cols.cast_ever,
    ]
    df_fact = df_saldos[fact_cols].copy()

    # Tipos
    df_fact[cols.mob] = pd.to_numeric(df_fact[cols.mob], errors="coerce").astype("Int64")
    if df_fact[cols.mob].isna().any():
        raise ValueError("Hay MOBs no numéricos o nulos en df_saldos.")

    # Numeradores a numérico, nulos a 0 (OK)
    for c in [cols.bgi2, cols.bgi3, cols.bgi4, cols.bgi5, cols.cast_val]:
        df_fact[c] = pd.to_numeric(df_fact[c], errors="coerce").fillna(0.0)

    # Denominador numérico; nulos a 0 (luego evitas dividir por 0)
    df_fact[cols.monto_fondeado] = pd.to_numeric(df_fact[cols.monto_fondeado], errors="coerce").fillna(0.0)

    # EVER flags 0/1
    ever_cols = [cols.bgi2_ever, cols.bgi3_ever, cols.bgi4_ever, cols.bgi5_ever, cols.cast_ever]
    for c in ever_cols:
        df_fact[c] = (pd.to_numeric(df_fact[c], errors="coerce").fillna(0) > 0).astype(np.int8)

    # --- 3) Merge dims ---
    df_master = df_fact.merge(df_dim, how="left", on=cols.folio, validate="m:1")

    # --- 4) Orden final (y solo columnas del contrato) ---
    ordered = [
        cols.folio, cols.mob, "cosecha", cols.monto_fondeado, cols.tipo_de_castigo, cols.estatus_legal,
        cols.bgi2, cols.bgi3, cols.bgi4, cols.bgi5, cols.cast_val,
        cols.bgi2_ever, cols.bgi3_ever, cols.bgi4_ever, cols.bgi5_ever, cols.cast_ever,
        "Perfil", "Ciudad Atria", "Plazo", "Antiguedad Unidad",
        "Laboratorios", "Notificaciones", "Atria Plus",
        "MMX 6", "MMX 12", "MMX 24",
        "Score generico de bancos", "Cluster Mora", "Cluster Experiencia",
        "Score Buro", "% Utilizacion Revolvente", "Endeudamiento",
    ]
    df_master = df_master[ordered]

    return df_master

In [76]:
df_saldos = pd.read_excel('Saldo_cosechas_20251201.xlsx')
df_saldos.head(20)

,folio,MOB,pv,Fecha de Inicio,Mes-Año,Monto Fondeado,Saldo Capital,Tipo de Castigo,Estatus Legal,BGI4+,...,BGI5+,BGI2+_CONTEO,BGI3+_CONTEO,BGI5+_CONTEO,BGI2+_CONTEO_EVER,BGI3+_CONTEO_EVER,BGI5+_CONTEO_EVER,CAST_VAL,CAST_CONTEO,CAST_VAL_ORI
0,111120220600071,1,0,2022-06-01,2022-07-01,107677.16,94927.53,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
1,111120220600071,2,0,2022-06-01,2022-08-01,107677.16,93876.24,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
2,111120220600071,3,0,2022-06-01,2022-09-01,107677.16,92797.52,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
3,111120220600071,4,0,2022-06-01,2022-10-01,107677.16,91690.63,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
4,111120220600071,5,0,2022-06-01,2022-11-01,107677.16,82523.46,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
5,111120220600071,6,0,2022-06-01,2022-12-01,107677.16,81461.39,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
6,111120220600071,7,0,2022-06-01,2023-01-01,107677.16,80371.61,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
7,111120220600071,8,0,2022-06-01,2023-02-01,107677.16,79253.39,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
8,111120220600071,9,0,2022-06-01,2023-03-01,107677.16,78105.98,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN
9,111120220600071,10,0,2022-06-01,2023-04-01,107677.16,76928.62,NaN,NaN,0.0,...,0.0,0,0,0,0,0,0,0.0,0,NaN


In [77]:
df_saldos.shape

(69395, 29)

In [78]:
df_master_dummy = build_df_master_dummy(df_saldos, seed=42)
df_master_dummy

,folio,MOB,cosecha,Monto Fondeado,Tipo de Castigo,Estatus Legal,BGI2+,BGI3+,BGI4+,BGI5+,...,Atria Plus,MMX 6,MMX 12,MMX 24,Score generico de bancos,Cluster Mora,Cluster Experiencia,Score Buro,% Utilizacion Revolvente,Endeudamiento
0,111120220600071,1,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
1,111120220600071,2,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
2,111120220600071,3,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
3,111120220600071,4,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
4,111120220600071,5,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69390,111120251002556,1,2025-10,277638.50,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP1,MOP1,MOP4,697,C0 - Sin mora,Exp5,556 a <613,>=0.305 a <0.621,<0.019
69391,111120251002572,1,2025-10,208019.56,NaN,NaN,0.0,0.0,0.0,0.0,...,PE,MOP1,MOP1,MOP2,666,C0 - Sin mora,Exp5,>663,>=0 a <0.068,>=0.171 a <0.305
69392,111120251002630,1,2025-10,222809.23,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP3,MOP4,MOP2,699,C0 - Sin mora,Exp5,>663,>=0.305 a <0.621,>=0.067 a <0.171
69393,111120251002639,1,2025-10,182272.09,NaN,NaN,0.0,0.0,0.0,0.0,...,Normal,MOP1,MOP2,MOP2,898,C0 - Sin mora,Exp2,>663,>=0.068 a <0.305,<0.019


In [79]:
df_master_dummy.columns

Index(['folio', 'MOB', 'cosecha', 'Monto Fondeado', 'Tipo de Castigo',
       'Estatus Legal', 'BGI2+', 'BGI3+', 'BGI4+', 'BGI5+', 'CAST_VAL',
       'BGI2+_CONTEO_EVER', 'BGI3+_CONTEO_EVER', 'BGI4+_CONTEO_EVER',
       'BGI5+_CONTEO_EVER', 'CAST_CONTEO', 'Perfil', 'Ciudad Atria', 'Plazo',
       'Antiguedad Unidad', 'Laboratorios', 'Notificaciones', 'Atria Plus',
       'MMX 6', 'MMX 12', 'MMX 24', 'Score generico de bancos', 'Cluster Mora',
       'Cluster Experiencia', 'Score Buro', '% Utilizacion Revolvente',
       'Endeudamiento'],
      dtype='object')

In [80]:
df_master_dummy.to_csv('output_data/df_master_dummy.csv', index=False)
df_prueba = pd.read_csv('output_data/df_master_dummy.csv')
df_prueba

,folio,MOB,cosecha,Monto Fondeado,Tipo de Castigo,Estatus Legal,BGI2+,BGI3+,BGI4+,BGI5+,...,Atria Plus,MMX 6,MMX 12,MMX 24,Score generico de bancos,Cluster Mora,Cluster Experiencia,Score Buro,% Utilizacion Revolvente,Endeudamiento
0,111120220600071,1,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
1,111120220600071,2,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
2,111120220600071,3,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
3,111120220600071,4,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
4,111120220600071,5,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69390,111120251002556,1,2025-10,277638.50,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP1,MOP1,MOP4,697,C0 - Sin mora,Exp5,556 a <613,>=0.305 a <0.621,<0.019
69391,111120251002572,1,2025-10,208019.56,NaN,NaN,0.0,0.0,0.0,0.0,...,PE,MOP1,MOP1,MOP2,666,C0 - Sin mora,Exp5,>663,>=0 a <0.068,>=0.171 a <0.305
69392,111120251002630,1,2025-10,222809.23,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP3,MOP4,MOP2,699,C0 - Sin mora,Exp5,>663,>=0.305 a <0.621,>=0.067 a <0.171
69393,111120251002639,1,2025-10,182272.09,NaN,NaN,0.0,0.0,0.0,0.0,...,Normal,MOP1,MOP2,MOP2,898,C0 - Sin mora,Exp2,>663,>=0.068 a <0.305,<0.019


In [93]:
print(df_prueba['Tipo de Castigo'].unique())
print(df_prueba['Estatus Legal'].unique())

[nan 'Mora' 'Administrativo' 'Operativo']
[nan 'Mora' 'Cancelación' 'Otros' 'Pérdida total con expediente'
 'Dación en pago' 'Fraude' 'Siniestro' 'Defunción del titular']


In [81]:
# ---------------------------
# Helpers para validar rápidamente en tu notebook
# ---------------------------
def compute_impago_matrix_exposure(
    df_master: pd.DataFrame,
    tipo_mora: str,
    filters: Optional[Dict[str, List[str]]] = None,
    cols: Cols = Cols(),
) -> pd.DataFrame:
    """
    Calcula matriz %impago por exposición:
      % = sum(BGI) / sum(Monto Fondeado) * 100
    df_master: 1 fila = (folio, MOB)
    tipo_mora: 'BGI2+'|'BGI3+'|'BGI4+'|'BGI5+'|'CAST_VAL'
    filters: dict {col: [allowed_values]}
    """
    df = df_master.copy()

    if filters:
        for k, v in filters.items():
            if k in df.columns and v is not None and len(v) > 0:
                df = df[df[k].isin(v)]

    if tipo_mora not in df.columns:
        raise ValueError(f"tipo_mora='{tipo_mora}' no existe como columna en df_master.")

    # Agregar por (cosecha, MOB)
    g = df.groupby(["cosecha", cols.mob], as_index=False).agg(
        numer=(tipo_mora, "sum"),
        denom=(cols.monto_fondeado, "sum"),
    )
    # Evitar división por 0
    g["pct_impago"] = np.where(g["denom"] > 0, (g["numer"] / g["denom"]) * 100.0, np.nan)

    mat = g.pivot(index="cosecha", columns=cols.mob, values="pct_impago").sort_index()
    return mat

In [82]:
mat = compute_impago_matrix_exposure(
    df_master_dummy,
    tipo_mora="BGI4+",
    filters={"Ciudad Atria": ["CDMX", "Monterrey"], "Plazo": ["36", "48"]}
)

In [83]:
mat

MOB,1,2,3,4,5,6,7,8,9,10,...,31,32,33,34,35,36,37,38,39,40
cosecha,,,,,,,,,,,,,,,,,,,,,
2022-07,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.0,0.0
2022-08,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.0,NaN
2022-09,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,7.697945,7.466723,7.09185,6.706298,NaN,NaN
2022-10,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,NaN,NaN,NaN
2022-11,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,9.804887,9.804887,9.804887,9.804887,9.804887,9.804887,NaN,NaN,NaN,NaN
2022-12,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
2023-01,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,18.355611,18.355611,17.820185,17.479460,NaN,NaN,NaN,NaN,NaN,NaN
2023-02,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-03,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,6.002606,6.002606,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [84]:
df_dummy = pd.read_csv('output_data/df_master_dummy.csv')
df_dummy.head(20).to_csv('output_data/df_master_dummy_extract.csv', index=False)


In [85]:
df_dummy = pd.read_csv('output_data/df_master_dummy.csv')
df_dummy.head(10)

,folio,MOB,cosecha,Monto Fondeado,Tipo de Castigo,Estatus Legal,BGI2+,BGI3+,BGI4+,BGI5+,...,Atria Plus,MMX 6,MMX 12,MMX 24,Score generico de bancos,Cluster Mora,Cluster Experiencia,Score Buro,% Utilizacion Revolvente,Endeudamiento
0,111120220600071,1,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
1,111120220600071,2,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
2,111120220600071,3,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
3,111120220600071,4,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
4,111120220600071,5,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
5,111120220600071,6,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
6,111120220600071,7,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
7,111120220600071,8,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
8,111120220600071,9,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171
9,111120220600071,10,2022-06,107677.16,NaN,NaN,0.0,0.0,0.0,0.0,...,GG,MOP4,MOP2,MOP2,804,C0 - Sin mora,Exp3,556 a <613,>=0.068 a <0.305,>=0.067 a <0.171


In [86]:
df_dummy.columns

Index(['folio', 'MOB', 'cosecha', 'Monto Fondeado', 'Tipo de Castigo',
       'Estatus Legal', 'BGI2+', 'BGI3+', 'BGI4+', 'BGI5+', 'CAST_VAL',
       'BGI2+_CONTEO_EVER', 'BGI3+_CONTEO_EVER', 'BGI4+_CONTEO_EVER',
       'BGI5+_CONTEO_EVER', 'CAST_CONTEO', 'Perfil', 'Ciudad Atria', 'Plazo',
       'Antiguedad Unidad', 'Laboratorios', 'Notificaciones', 'Atria Plus',
       'MMX 6', 'MMX 12', 'MMX 24', 'Score generico de bancos', 'Cluster Mora',
       'Cluster Experiencia', 'Score Buro', '% Utilizacion Revolvente',
       'Endeudamiento'],
      dtype='object')

In [113]:
cosecha = '2022-10'

df_dummy = pd.read_csv('output_data/df_master_dummy.csv')
df_dummy_1 = df_dummy[(df_dummy['cosecha'] == f'{cosecha}') & (df_dummy['MOB'] == 12) & (df_dummy['BGI4+_CONTEO_EVER'] == 1)][['folio', 'cosecha', 'MOB', 'BGI4+_CONTEO_EVER', 'BGI4+']]
cont_bgi4ever = df_dummy_1['BGI4+'].sum()
print(cont_bgi4ever)


df_dummy_2 = df_dummy[(df_dummy['cosecha'] == f'{cosecha}') & (df_dummy['MOB'] == 1)]
cont_total = df_dummy_2['Monto Fondeado'].sum()
print(cont_total)

print(round((cont_bgi4ever/cont_total)*100, 2))

5043.460000000021
5616771.100000001
0.09


In [99]:
[1,2] == [1, 2]

True

In [ ]:
import pandas as pd
df_dummy = pd.read_csv('output_data/df_master_dummy.csv')

v_reales = [107677, 915900, 3182186, 5472540, 5417393, 5071811, 7175036, 10530727, 11923880, 17340275, 13615420, 18523038, 18950168, 20423491, 17385508, 19237983, 19775606, 13958174, 12409313, 12438942, 12457552, 9806875, 11010323, 15072724, 13505108, 11352063, 17868903, 13089272, 20168756, 18741434, 18246983, 20721273, 17431239, 18549881, 22286096, 29682321, 32119365, 43911858, 45226287, 49119639, 57091652]

cosechas = pd.period_range('2022-06', '2025-10', freq='M').astype(str).tolist()
# cosechas

filtros = ['Mora', 'Administrativo', 'Operativo']
filtros_n = [
    [filtros[0]],
    [filtros[1]],
    [filtros[2]],
    [filtros[0], filtros[1]],
    [filtros[0], filtros[2]],
    [filtros[1], filtros[2]],
    filtros
]

for filtro in filtros_n:
    v_calc = []
    for cosecha in cosechas:
        df_1 = df_dummy[(df_dummy['cosecha'] == cosecha) & (df_dummy['MOB'] == 1)]
        df_2 = df_1[df_1['Tipo de Castigo'].isin(filtro) == False]
        v_calc.append(int(round(df_2['Monto Fondeado'].sum(), 0)))

    if v_calc == v_reales:
        print("Coincide con filtro:", filtro)
        break
    else:
        print(v_calc)
        print("No se encontró filtro que coincida.")



# 107677, 915900, 3182186, 5472540, 5417393, 5071811, 7175036, 10530727, 11923880, 17340275, 13615420, 18523038, 18950168, 20423491, 17385508, 19237983, 19775606, 13958174, 12409313, 12438942, 12457552, 9806875, 11010323, 15072724, 13505108, 11352063, 17868903, 13089272, 20168756, 18741434, 18246983, 20721273, 17431239, 18549881, 22286096, 29682321, 32119365, 43911858, 45226287, 49119639, 57091652


[107677, 915900, 3182186, 5354488, 5417393, 5071811, 7175036, 10246819, 11923880, 17358215, 13324912, 18527319, 18970198, 20286469, 17237877, 19381403, 20016474, 14008228, 12622274, 12405084, 12107589, 9983787, 11010323, 14920643, 13505108, 11061543, 17862880, 12961780, 20118099, 19043060, 18104014, 20335156, 17268585, 18549881, 22177944, 29682321, 32119365, 44062098, 45226287, 49494911, 57091652]
No se encontró filtro que coincida.
[107677, 915900, 3182186, 5472540, 5616771, 5071811, 7175036, 10530727, 11923880, 17340275, 13615420, 18523038, 18950168, 20423491, 17385508, 19237983, 20294747, 13958174, 12409313, 12438942, 12457552, 9924843, 11010323, 15072724, 13505108, 11352063, 17868903, 13089272, 20168756, 18741434, 18246983, 20721273, 17431239, 18549881, 22286096, 29682321, 32119365, 43911858, 45226287, 49119639, 57091652]
No se encontró filtro que coincida.
[107677, 915900, 3182186, 5354488, 5417393, 5071811, 7175036, 10246819, 11923880, 17134553, 13136922, 18358238, 18769311, 2028

In [ ]:
import pandas as pd
df_dummy = pd.read_csv('output_data/df_master_dummy.csv')

v_reales = [107677, 915900, 3182186, 5472540, 5417393, 5071811, 7175036, 10530727, 11923880, 17340275, 13615420, 18523038, 18950168, 20423491, 17385508, 19237983, 19775606, 13958174, 12409313, 12438942, 12457552, 9806875, 11010323, 15072724, 13505108, 11352063, 17868903, 13089272, 20168756, 18741434, 18246983, 20721273, 17431239, 18549881, 22286096, 29682321, 32119365, 43911858, 45226287, 49119639, 57091652]

cosechas = pd.period_range('2022-06', '2025-10', freq='M').astype(str).tolist()
# cosechas

filtros = ['Fraude', 'Otros']
filtros_n = [
    filtros
]
# filtros_n

for filtro in filtros_n:
    v_calc = []
    for cosecha in cosechas:
        df_1 = df_dummy[(df_dummy['cosecha'] == cosecha) & (df_dummy['MOB'] == 1)]
        df_2 = df_1[df_1['Estatus Legal'].isin(filtro) == False]
        v_calc.append(int(round(df_2['Monto Fondeado'].sum(), 0)))

    if v_calc == v_reales:
        print("Coincide con filtro:", filtro)
        break
    else:
        print(v_calc)
        print("No se encontró filtro que coincida.")

Coincide con filtro: ['Fraude', 'Otros']
